In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.model_selection import GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [28]:
Dataset = pd.read_csv(r'../backend/screen_time_mental_health.csv')
Dataset = Dataset[["sex", "sleep_quality_index", "avg_sleep_hours", "bdi_total"]]
Dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 4810 entries, 0 to 4809
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sex                  4810 non-null   str    
 1   sleep_quality_index  4810 non-null   float64
 2   avg_sleep_hours      4810 non-null   float64
 3   bdi_total            4810 non-null   int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 150.4 KB


In [29]:
males_data = Dataset[Dataset["sex"] == "Boy"].drop(columns=["sex"])
females_data = Dataset[Dataset["sex"] == "Girl"].drop(columns=["sex"])
Dataset = Dataset.drop(columns=["sex"])
males_data.info()
females_data.info()

<class 'pandas.DataFrame'>
Index: 2446 entries, 0 to 4809
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sleep_quality_index  2446 non-null   float64
 1   avg_sleep_hours      2446 non-null   float64
 2   bdi_total            2446 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 76.4 KB
<class 'pandas.DataFrame'>
Index: 2364 entries, 1 to 4808
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   sleep_quality_index  2364 non-null   float64
 1   avg_sleep_hours      2364 non-null   float64
 2   bdi_total            2364 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 73.9 KB


In [30]:
X = Dataset[["sleep_quality_index", "avg_sleep_hours"]]
Y = Dataset["bdi_total"]

Xf = females_data[["sleep_quality_index", "avg_sleep_hours"]]
Yf = females_data["bdi_total"]

Xm = males_data[["sleep_quality_index", "avg_sleep_hours"]]
Ym = males_data["bdi_total"]

In [31]:
# Train/test split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=20)
Xf_train, Xf_test, Yf_train, Yf_test = train_test_split(Xf, Yf, test_size=0.2, random_state=20)
Xm_train, Xm_test, Ym_train, Ym_test = train_test_split(Xm, Ym, test_size=0.2, random_state=20)

In [41]:
def train_models(X_train, Y_train):
    cv = KFold(n_splits=5, shuffle=True, random_state=20)

    #Linear Regression
    lr_model = Pipeline([
        ('scaler', StandardScaler()),
        ('poly', PolynomialFeatures(include_bias=False)),
        ('reg', LinearRegression())
    ])
    lr_params = {'poly__degree': [1, 2, 3]}
    lr_grid = GridSearchCV(lr_model, lr_params, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    lr_grid.fit(X_train, Y_train)
    print(f"Ideal Polynomial: {lr_grid.best_params_['poly__degree']}")

    #SVM
    svm_model = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVR())
    ])
    svm_params = {
        'svm__kernel': ['rbf'],
        'svm__C': [1, 10, 100],
        'svm__epsilon': [0.1, 0.5, 1.0]
    }
    svm_grid = GridSearchCV(svm_model, svm_params, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    svm_grid.fit(X_train, Y_train)

    #Random Forest
    rf_model = RandomForestRegressor(random_state=20)
    rf_params = {
        'n_estimators': [200],
        'max_depth': [None, 10, 20],
        'min_samples_leaf': [1, 2, 4]
    }
    rf_grid = GridSearchCV(rf_model, rf_params, cv=cv, scoring='neg_mean_squared_error', n_jobs=-1)
    rf_grid.fit(X_train, Y_train)

    #Ensamble
    ensemble = VotingRegressor([
        ('polynomial', lr_grid.best_estimator_),
        ('svm', svm_grid.best_estimator_),
        ('random_forest', rf_grid.best_estimator_)
    ])
    ensemble.fit(X_train, Y_train)

    return {
        'polynomial': lr_grid,
        'svm': svm_grid,
        'random_forest': rf_grid,
        'ensemble': ensemble,
    }


In [42]:
def evaluate_models(models, X_test, Y_test):
    results = {}

    for name, model in models.items():
        predictions = model.predict(X_test)

        results[name] = {
            'MAE': mean_absolute_error(Y_test, predictions),
            'RMSE': root_mean_squared_error(Y_test, predictions),
            'R²': r2_score(Y_test, predictions)
        }

    return results

In [43]:
models = train_models(X_train, Y_train)
female_models = train_models(Xf_train, Yf_train)
male_models = train_models(Xm_train, Ym_train)

unsorted_results = evaluate_models(models, X_test, Y_test)
female_results = evaluate_models(female_models, Xf_test, Yf_test)
male_results = evaluate_models(male_models, Xm_test, Ym_test)

unsorted_results_df = pd.DataFrame(unsorted_results).T
female_results_df = pd.DataFrame(female_results).T
male_results_df = pd.DataFrame(male_results).T

Ideal Polynomial: 2
Ideal Polynomial: 2
Ideal Polynomial: 1


In [44]:
all_results = pd.concat(
    [unsorted_results_df, female_results_df, male_results_df],
    keys=['All', 'Female', 'Male']
)
all_results

MAE      RMSE        R²
All    polynomial     5.054654  7.012057  0.178578
       svm            4.842507  7.292515  0.111556
       random_forest  5.127567  7.141793  0.147901
       ensemble       4.937130  7.052162  0.169155
Female polynomial     5.691031  7.792843  0.170413
       svm            5.568303  8.105891  0.102424
       random_forest  5.814751  7.894531  0.148622
       ensemble       5.595353  7.806735  0.167453
Male   polynomial     3.772306  5.411310  0.121547
       svm            3.550429  5.550856  0.075656
       random_forest  3.838916  5.507476  0.090047
       ensemble       3.636930  5.368353  0.135438